# Gemma 4 E4B QLoRA Fine-Tuning

Dieses Notebook erwartet einen bereits fuer Gemma 4 vorbereiteten JSONL-Datensatz aus `util/convert_gdpr_process_format_to_json.py`.

Empfohlene Reihenfolge:
1. `python util/convert_gdpr_process_format_to_json.py`
2. Danach dieses Notebook von oben nach unten ausfuehren.

Wichtige Punkte:
- Das Umschreiben in das Gemma-4-Trainingsformat passiert nicht mehr im Notebook.
- Das Notebook laedt nur noch den fertigen Datensatz, splittet ihn in `train` / `validation` / `test` und trainiert darauf.
- Die Trainingsparameter sind bewusst auf euren kleinen One-Shot-Use-Case mit ca. 50 Beispielen und einer RTX 4090 zugeschnitten.


In [ ]:
%pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
%pip install -U "transformers>=5.6.0" "accelerate>=1.13.0" "datasets>=4.8.0" "trl>=0.23.0" "peft>=0.19.0" "bitsandbytes>=0.49.0" "sentencepiece>=0.2.0" "protobuf>=5.28.0" "huggingface_hub>=0.34.0" "tensorboard>=2.20.0"


In [ ]:
import torch
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
print("bitsandbytes:", bnb.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch wurde ohne funktionierende CUDA-Unterstuetzung geladen. Bitte die Install-Zelle pruefen.")


In [ ]:
import gc
import json
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from datasets import Dataset, DatasetDict
from huggingface_hub import login
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForImageTextToText, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback, TrainerCallback, pipeline, set_seed
from trl import SFTConfig, SFTTrainer

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
set_seed(42)

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"gpu: {props.name}")
    print(f"compute capability: {torch.cuda.get_device_capability(0)}")
    print(f"vram: {props.total_memory / 1024**3:.2f} GB")


In [ ]:
DATA_PATH = Path("training_data_LLM_format")
OUTPUT_DIR = Path("gemma4_legal_process_lora") / "e4b_gdpr_qlora_adapter_one_shot"
MERGED_OUTPUT_DIR = Path("gemma4_legal_process_lora") / "e4b_gdpr_qlora_merged_one_shot"

BASE_MODEL_ID = "google/gemma-4-E4B"
TOKENIZER_ID = "google/gemma-4-E4B-it"

SEED = 42
TRAIN_SPLIT = 0.85
VALIDATION_SPLIT = 0.10
TEST_SPLIT = 0.05

# Auf 94 Beispielen ist eine konservative Einzellauf-Konfiguration sinnvoll:
# - Kontext wird datengetrieben bestimmt, aber aus Laufzeitgruenden hart bei 8192 gedeckelt
# - fuer den strukturierten XML-Use-Case etwas weniger Regularisierung, damit das Format straffer gelernt wird
# - haeufigere Eval/Save-Schritte, damit ein einmaliger Lauf besser beobachtbar und frueher stoppbar ist
MAX_SEQ_LENGTH = "auto_fit_dataset"
AUTO_MAX_SEQ_HEADROOM = 512
AUTO_MAX_SEQ_ROUND_TO = 1024
AUTO_MAX_SEQ_HARD_CAP = 8192
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

NUM_TRAIN_EPOCHS = 10
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 0.2
EARLY_STOPPING_PATIENCE = 6
TRAIN_EXAMPLE_PROGRESS_EVERY = 1
LOGGING_STEPS = 1
EVAL_SAVE_STEPS = 10
SAVE_TOTAL_LIMIT = 3
INFERENCE_MAX_NEW_TOKENS = 8192

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH = {DATA_PATH.resolve()}")
print(f"OUTPUT_DIR = {OUTPUT_DIR.resolve()}")
print(f"BASE_MODEL_ID = {BASE_MODEL_ID}")
print(f"TOKENIZER_ID = {TOKENIZER_ID}")


In [ ]:
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=True)
    print("Hugging Face Login ueber HF_TOKEN aus der Umgebung erfolgreich.")
else:
    print("Kein HF_TOKEN gefunden. Training und lokale Inferenz koennen trotzdem funktionieren, solange Modell und Tokenizer anonym geladen werden duerfen.")
    print("Ohne Login sind Downloads oft langsamer und unterliegen niedrigeren Rate Limits. Fuer diesen lokalen Lauf ist Login aber optional.")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(tokenizer.special_tokens_map)
print("Chat template vorhanden:", tokenizer.chat_template is not None)


In [ ]:
def load_jsonl(path: Path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Ungueltiges JSON in Zeile {line_number}: {exc}") from exc
    return records


def validate_record(record, line_number):
    for field in ["article_id", "prompt", "completion", "full_text"]:
        value = record.get(field)
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f"Feld '{field}' fehlt oder ist leer in Datensatz-Zeile {line_number}.")

    if record["full_text"] != record["prompt"] + record["completion"]:
        raise ValueError(f"prompt + completion stimmt nicht mit full_text ueberein in Zeile {line_number}.")

    source_messages = record.get("source_messages")
    if source_messages is not None:
        if not isinstance(source_messages, list) or len(source_messages) < 3:
            raise ValueError(f"source_messages ist unvollstaendig in Zeile {line_number}.")
        roles = [message.get("role") for message in source_messages]
        if roles[:2] != ["system", "user"] or roles[-1] != "assistant":
            raise ValueError(f"Unerwartete Rollenfolge in source_messages, Zeile {line_number}: {roles}")

    if "full_text_token_count" not in record:
        record["full_text_token_count"] = len(tokenizer(record["full_text"], add_special_tokens=False)["input_ids"])

    return record


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Datensatz nicht gefunden: {DATA_PATH}. Bitte zuerst util/convert_gdpr_process_format_to_json.py ausfuehren."
    )

raw_records = load_jsonl(DATA_PATH)
records = [validate_record(record, idx) for idx, record in enumerate(raw_records, start=1)]

print(f"Anzahl Beispiele: {len(records)}")
print(f"Erstes article_id: {records[0]['article_id']}")
print("Prompt-Vorschau:\n")
print(records[0]["prompt"][:1500])
print("\nCompletion-Vorschau:\n")
print(records[0]["completion"][:1200])


In [ ]:
if not math.isclose(TRAIN_SPLIT + VALIDATION_SPLIT + TEST_SPLIT, 1.0, rel_tol=0.0, abs_tol=1e-9):
    raise ValueError("TRAIN_SPLIT + VALIDATION_SPLIT + TEST_SPLIT muss 1.0 ergeben.")

holdout_split = VALIDATION_SPLIT + TEST_SPLIT
test_share_of_holdout = TEST_SPLIT / holdout_split

dataset = Dataset.from_list(records).shuffle(seed=SEED)
train_holdout = dataset.train_test_split(test_size=holdout_split, seed=SEED)
validation_test = train_holdout["test"].train_test_split(test_size=test_share_of_holdout, seed=SEED)

dataset = DatasetDict(
    {
        "train": train_holdout["train"],
        "validation": validation_test["train"],
        "test": validation_test["test"],
    }
)

print(dataset)
print({split: len(dataset[split]) for split in dataset})
print(f"Split-Ziel: train={TRAIN_SPLIT:.0%}, validation={VALIDATION_SPLIT:.0%}, test={TEST_SPLIT:.0%}")


In [ ]:
split_rows = []
all_lengths = []

for split_name, split_dataset in dataset.items():
    split_lengths = [int(row["full_text_token_count"]) for row in split_dataset]
    all_lengths.extend(split_lengths)
    split_array = np.array(split_lengths)
    split_rows.append(
        {
            "split": split_name,
            "count": int(split_array.size),
            "min": int(split_array.min()),
            "median": int(np.median(split_array)),
            "p90": int(np.quantile(split_array, 0.90)),
            "p95": int(np.quantile(split_array, 0.95)),
            "max": int(split_array.max()),
        }
    )

length_df = pd.DataFrame(split_rows)
display(length_df)

all_lengths = np.array(all_lengths)
dataset_max_length = int(all_lengths.max())

if MAX_SEQ_LENGTH == "auto_fit_dataset":
    resolved_max_seq_length = int(
        math.ceil((dataset_max_length + AUTO_MAX_SEQ_HEADROOM) / AUTO_MAX_SEQ_ROUND_TO) * AUTO_MAX_SEQ_ROUND_TO
    )
elif isinstance(MAX_SEQ_LENGTH, str):
    raise ValueError(f"Unbekannte MAX_SEQ_LENGTH-Einstellung: {MAX_SEQ_LENGTH}")
else:
    resolved_max_seq_length = int(MAX_SEQ_LENGTH)

if AUTO_MAX_SEQ_HARD_CAP is not None:
    resolved_max_seq_length = min(resolved_max_seq_length, int(AUTO_MAX_SEQ_HARD_CAP))

truncation_share = float((all_lengths > resolved_max_seq_length).mean() * 100)
effective_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
updates_per_epoch = math.ceil(len(dataset["train"]) / effective_batch_size)
estimated_total_updates = updates_per_epoch * NUM_TRAIN_EPOCHS

print(f"Laengstes Beispiel im Datensatz: {dataset_max_length}")
print(f"Verwendete MAX_SEQ_LENGTH: {resolved_max_seq_length}")
print(f"Geschaetzter Anteil truncierter Beispiele: {truncation_share:.2f}%")
print(f"Effektive Batchgroesse: {effective_batch_size}")
print(f"Geschaetzte Optimizer-Updates pro Epoche: {updates_per_epoch}")
print(f"Geschaetzte Gesamtzahl Optimizer-Updates: {estimated_total_updates}")


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Dieses Notebook braucht fuer QLoRA eine CUDA-faehige NVIDIA-GPU.")

major, _ = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if major >= 8 else torch.float16
print(f"compute_dtype = {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_storage=compute_dtype,
)

model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    dtype=compute_dtype,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    quantization_config=bnb_config,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

print(type(model).__name__)
print(model.config.model_type)


In [ ]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"],
    ensure_weight_tying=True,
)

peft_config


In [ ]:
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    max_length=resolved_max_seq_length,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    save_strategy="steps",
    save_steps=EVAL_SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=MAX_GRAD_NORM,
    completion_only_loss=True,
    packing=False,
    dataloader_num_workers=0,
    disable_tqdm=False,
    report_to="tensorboard",
    seed=SEED,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": False,
    },
)

training_args


In [ ]:
train_examples = len(dataset["train"])
effective_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
optimizer_steps_per_epoch = math.ceil(train_examples / effective_batch_size)


def format_seconds(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


class ConsoleProgressCallback(TrainerCallback):
    def __init__(self, train_examples, optimizer_steps_per_epoch, gradient_accumulation_steps, print_every_train_example=1):
        self.train_examples = max(1, int(train_examples))
        self.optimizer_steps_per_epoch = max(1, int(optimizer_steps_per_epoch))
        self.gradient_accumulation_steps = max(1, int(gradient_accumulation_steps))
        self.print_every_train_example = max(1, int(print_every_train_example))
        self.train_start_time = None
        self.examples_seen = 0

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start_time = time.time()
        self.examples_seen = 0
        print(
            f"Training startet: max_steps={state.max_steps}, "
            f"train_examples_per_epoch={self.train_examples}, "
            f"optimizer_steps_per_epoch={self.optimizer_steps_per_epoch}, "
            f"geplante epochen={args.num_train_epochs}"
        )
        return control

    def _print_example_progress(self, args, state):
        if self.examples_seen % self.print_every_train_example != 0:
            return

        elapsed = time.time() - self.train_start_time if self.train_start_time is not None else 0.0
        avg_example_time = elapsed / max(1, self.examples_seen)
        total_examples_planned = self.train_examples * int(math.ceil(args.num_train_epochs))
        remaining_examples = max(0, total_examples_planned - self.examples_seen)
        eta_seconds = remaining_examples * avg_example_time

        epoch_number = ((self.examples_seen - 1) // self.train_examples) + 1
        epoch_example = ((self.examples_seen - 1) % self.train_examples) + 1
        epoch_progress = 100.0 * epoch_example / self.train_examples
        accumulation_progress = ((epoch_example - 1) % self.gradient_accumulation_steps) + 1
        optimizer_step_in_epoch = math.ceil(epoch_example / self.gradient_accumulation_steps)
        total_epoch_count = int(math.ceil(args.num_train_epochs))
        total_progress = 100.0 * self.examples_seen / max(1, total_examples_planned)

        print(
            f"Epoch {epoch_number}/{total_epoch_count} | "
            f"example {epoch_example}/{self.train_examples} ({epoch_progress:5.1f}%) | "
            f"accum {accumulation_progress}/{self.gradient_accumulation_steps} | "
            f"optimizer-step {optimizer_step_in_epoch}/{self.optimizer_steps_per_epoch} | "
            f"global-step {state.global_step}/{state.max_steps} | "
            f"total {self.examples_seen}/{total_examples_planned} ({total_progress:5.1f}%) | "
            f"elapsed {format_seconds(elapsed)} | ETA {format_seconds(eta_seconds)}"
        )

    def on_substep_end(self, args, state, control, **kwargs):
        self.examples_seen += 1
        self._print_example_progress(args, state)
        return control

    def on_step_end(self, args, state, control, **kwargs):
        self.examples_seen += 1
        self._print_example_progress(args, state)
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        print(f"Evaluation gestartet nach Epoch {state.epoch:.2f}.")
        if metrics:
            print(metrics)
        return control

    def on_save(self, args, state, control, **kwargs):
        checkpoint_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        print(f"Checkpoint gespeichert: {checkpoint_dir}")
        return control


trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=peft_config,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE),
        ConsoleProgressCallback(
            train_examples=train_examples,
            optimizer_steps_per_epoch=optimizer_steps_per_epoch,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            print_every_train_example=TRAIN_EXAMPLE_PROGRESS_EVERY,
        ),
    ],
)

trainer


In [ ]:
train_result = trainer.train()
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

train_result


In [ ]:
test_metrics = trainer.evaluate(eval_dataset=dataset["test"])
test_metrics


## Optional: Adapter in das Basismodell mergen

Hinweis: Dieser Schritt braucht deutlich mehr CPU-RAM als das reine Adapter-Training.
Wenn ihr nur den LoRA-Adapter weiterverwenden wollt, koennt ihr diesen Schritt ueberspringen.


In [ ]:
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    dtype=compute_dtype,
    low_cpu_mem_usage=True,
)
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = peft_model.merge_and_unload()

MERGED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(MERGED_OUTPUT_DIR, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)

print(f"Merged model gespeichert unter: {MERGED_OUTPUT_DIR.resolve()}")


In [ ]:
sample = dataset["test"][0]
prompt_text = sample["prompt"]

if MERGED_OUTPUT_DIR.exists() and any(MERGED_OUTPUT_DIR.iterdir()):
    inference_model = AutoModelForImageTextToText.from_pretrained(
        MERGED_OUTPUT_DIR,
        device_map="auto",
        dtype="auto",
    )
else:
    base_model = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL_ID,
        device_map="auto",
        dtype="auto",
    )
    inference_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

pipe = pipeline("text-generation", model=inference_model, tokenizer=tokenizer)
generated = pipe(prompt_text, max_new_tokens=INFERENCE_MAX_NEW_TOKENS, do_sample=False)[0]["generated_text"]
prediction = generated[len(prompt_text):].strip()

print("PROMPT:\n")
print(prompt_text[:2000])
print("\nREFERENZ:\n")
print(sample["completion"][:2000])
print("\nMODELLAUSGABE:\n")
print(prediction[:4000])
